# Docking with cmxflow: blind, constrained, and scaffold-indexed

This notebook showcases the `cmxflow.operators.dock` subpackage on a real kinase
target (Abl1), using a **congeneric series** built around the co-crystallized
ligand. Three docking modes are demonstrated, all on the *same* set of molecules
and all running in parallel:

| Section | Mode | What it shows |
|---|---|---|
| **(a)** | **Blind docking** | Dock unaligned conformers; the block recenters on the reference pocket automatically. Iterated local search (ILS) finds lower-energy poses. |
| **(b)** | **Constrained docking** | Pin a known binding core to a template pose; only the substituents are optimized. |
| **(c)** | **Scaffold-indexed docking** | Automate (b) across the whole series via a cached scaffold pose — same quality, a fraction of the time. |

All timings are measured live, and the final docked poses are written to SDF for
review. In a production screen this dock stage is the last step of the full prep
pipeline (`standardize → enumerate stereo → confgen → ionize → align → dock`, see
`scripts/enrichment.py`); here we feed prepared conformers directly so the docking
features stay front-and-centre.

In [ ]:
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw

from cmxflow import Workflow
from cmxflow.operators.dock import (
    MoleculeDockBlock,
    scaffold_key,
    scaffold_pose,
    transfer_template_pose,
)
from cmxflow.sinks import MoleculeSinkBlock
from cmxflow.sources import MoleculeSourceBlock
from cmxflow.sources.reader import read_molecules
from cmxflow.utils.parallel import make_parallel

RDLogger.DisableLog("rdApp.*")

# Receptor + reference ligand ship one directory up, in examples/.
RECEPTOR = Path("../receptor.pdb").resolve()
REFERENCE = Path("../crystal_ligand.sdf").resolve()
assert RECEPTOR.exists() and REFERENCE.exists(), "run from examples/docking/"

OUT = Path(".")            # docked SDFs are written here for review
MAX_WORKERS = 8
print("receptor :", RECEPTOR.name)
print("reference:", REFERENCE.name)

## The system and a congeneric series

The Abl1 reference ligand is a pyrimidine-fused pyridinone kinase inhibitor. Its
Bemis–Murcko scaffold (the fused bicyclic hinge-binder plus the two pendant aryl
rings) is what *defines* the series: every substituent we vary below sits **off**
those ring systems — on the two phenyls or the lactam nitrogen — so all 13
molecules collapse to the **same scaffold key**. That shared scaffold is exactly
what the constrained and indexed modes exploit.

In [ ]:
reference_mol = next(read_molecules(REFERENCE, wrap=False))
print("reference SMILES:", Chem.MolToSmiles(Chem.RemoveAllHs(reference_mol)))
print("scaffold key    :", scaffold_key(reference_mol))

In [ ]:
# A 13-member congeneric series: the crystal ligand plus decorations of the two
# pendant phenyls and the lactam N. All share one Bemis-Murcko scaffold.
SERIES = {
    "crystal_3Me_4F": "Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)ccc1F",
    "aniline_H":      "c1ccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)cc1",
    "aniline_4F":     "Fc1ccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)cc1",
    "aniline_4Cl":    "Clc1ccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)cc1",
    "aniline_4OMe":   "COc1ccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)cc1",
    "aniline_3CF3":   "FC(F)(F)c1cccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)c1",
    "aniline_4Me":    "Cc1ccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)cc1",
    "lactam_NEt":     "Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(CC)c3n2)ccc1F",
    "lactam_NH":      "Cc1cc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)[nH]c3n2)ccc1F",
    "aryl_2Cl":       "Cc1cc(Nc2ncc3cc(-c4ccccc4Cl)c(=O)n(C)c3n2)ccc1F",
    "aryl_phenyl":    "Cc1cc(Nc2ncc3cc(-c4ccccc4)c(=O)n(C)c3n2)ccc1F",
    "aryl_26diF":     "Cc1cc(Nc2ncc3cc(-c4c(F)cccc4F)c(=O)n(C)c3n2)ccc1F",
    "aniline_34diF":  "Fc1ccc(Nc2ncc3cc(-c4c(Cl)cccc4Cl)c(=O)n(C)c3n2)cc1F",
}


def embed(name, smiles):
    '''Prepare a single 3D conformer (unaligned to the pocket).'''
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=2026)
    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp("_Name", name)
    return mol


congeners = [embed(n, s) for n, s in SERIES.items()]
keys = {scaffold_key(m) for m in congeners}
print(f"{len(congeners)} congeners, {len(keys)} unique scaffold key(s):")
for k in keys:
    print("  ", k)

In [ ]:
# A quick look at the series (2D).
Draw.MolsToGridImage(
    [Chem.RemoveAllHs(m) for m in congeners],
    legends=list(SERIES),
    molsPerRow=4,
    subImgSize=(260, 200),
)

In [ ]:
# Stage the unaligned conformers as the shared docking input. Kept in a scratch
# dir so the only artifacts left behind are the docked-pose SDFs.
SCRATCH = Path("_scratch")
SCRATCH.mkdir(exist_ok=True)
INPUT_SDF = SCRATCH / "congeners.sdf"
writer = Chem.SDWriter(str(INPUT_SDF))
for m in congeners:
    writer.write(m)
writer.close()

# Light search settings so every section runs in well under a minute.
SEARCH = dict(
    n_starts=8, max_distance_geometry_samples=8, sobol_max_tries=512, box_size=8.0
)


def dock_workflow(**dock_kwargs):
    '''Source -> parallel dock -> sink. Extra kwargs go to the dock block.'''
    wf = Workflow(name="dock")
    wf.add(
        MoleculeSourceBlock(),
        make_parallel(
            MoleculeDockBlock(
                receptor=str(RECEPTOR), site_reference=str(REFERENCE), **dock_kwargs
            ),
            max_workers=MAX_WORKERS,
        ),
        MoleculeSinkBlock(),
    )
    return wf


def run_dock(out_name, *, src=INPUT_SDF, **dock_kwargs):
    '''Dock ``src`` through the parallel workflow; return path, mols, seconds.'''
    out_path = OUT / out_name
    t0 = time.perf_counter()
    dock_workflow(**dock_kwargs)(str(src), str(out_path))
    seconds = time.perf_counter() - t0
    mols = [
        m for m in Chem.SDMolSupplier(str(out_path), removeHs=False) if m is not None
    ]
    return out_path, mols, seconds


def scores(mols):
    return np.array([float(m.GetProp("docking_score")) for m in mols])

## (a) Blind docking with automatic pocket recentering

The conformers above were embedded in vacuum — they are **not** positioned in the
binding site. Because we pass a `site_reference`, the dock block anchors its
search box on the reference ligand's centroid and seeds restarts with fresh
distance-geometry conformers placed in the pocket. So a raw conformer is enough;
no manual alignment step is required.

In [ ]:
blind_path, blind_mols, blind_t = run_dock("docked_blind.sdf", basin_hops=0)
print(f"blind dock: {len(blind_mols)} poses in {blind_t:.1f}s  "
      f"(mean score {scores(blind_mols).mean():.2f} kcal/mol)  -> {blind_path}")

### Better poses with iterated local search (ILS)

`basin_hops` turns on iterated local search: after each restart minimizes, the
pose is perturbed and re-minimized, letting it hop between nearby basins. It costs
more time but reliably finds lower-energy poses than the single-minimize default.

In [ ]:
ils_path, ils_mols, ils_t = run_dock("docked_blind_ils.sdf", basin_hops=6)

cmp = pd.DataFrame({
    "name": list(SERIES),
    "blind": scores(blind_mols).round(2),
    "blind+ILS": scores(ils_mols).round(2),
})
cmp["delta"] = (cmp["blind+ILS"] - cmp["blind"]).round(2)
improvement = scores(ils_mols).mean() - scores(blind_mols).mean()
print(f"default {blind_t:.1f}s (mean {scores(blind_mols).mean():.2f})   "
      f"ILS {ils_t:.1f}s (mean {scores(ils_mols).mean():.2f})   "
      f"mean improvement {improvement:+.2f} kcal/mol")
cmp

## (b) Constrained docking: pin a known binding core

If we already know how the scaffold binds — here, from the crystal structure — we
can **hold that core fixed** and optimize only the substituents. Two pieces:

1. **`transfer_template_pose`** places each congener's scaffold onto the crystal
   pose (substructure match → rigid alignment → snap → MMFF cleanup with the core
   held). This is the public template-docking primitive.
2. The dock block's **`constraint_smarts`** then restrains the matched core atoms
   to those positions during the search (a flat-bottom harmonic restraint), and
   automatically forces `n_starts → 1` — a single constrained local search.

The result: every pose shares the crystal's hinge geometry exactly, so the series
is mutually consistent.

In [ ]:
# Fused pyrimidine-pyridinone core, by element/bond so it matches every congener
# regardless of aromaticity perception or the lactam N substituent (Me / Et / H).
CORE_SMARTS = "[#6]1[#7][#6][#6]2[#6][#6][#6](=[#8])[#7][#6]2[#7]1"
core_pattern = Chem.MolFromSmarts(CORE_SMARTS)
core_template = scaffold_pose(reference_mol)  # Murcko scaffold carrying crystal coords

# Crystal core atom positions, to measure how tightly the constraint holds.
ref_heavy = Chem.RemoveAllHs(reference_mol)
ref_core_idx = list(ref_heavy.GetSubstructMatch(core_pattern))
crystal_core_xyz = np.array(ref_heavy.GetConformer().GetPositions())[ref_core_idx]


def core_rmsd_to_crystal(mol):
    heavy = Chem.RemoveAllHs(mol)
    idx = list(heavy.GetSubstructMatch(core_pattern))
    xyz = np.array(heavy.GetConformer().GetPositions())[idx]
    return float(np.sqrt(((xyz - crystal_core_xyz) ** 2).sum(1).mean()))


# Stage template-placed inputs, then constrained-dock them in parallel.
placed_sdf = SCRATCH / "congeners_placed.sdf"
writer = Chem.SDWriter(str(placed_sdf))
for mol in congeners:
    placed, _ = transfer_template_pose(mol, core_template)
    placed.SetProp("_Name", mol.GetProp("_Name"))
    writer.write(placed)
writer.close()

con_path, con_mols, con_t = run_dock(
    "docked_constrained.sdf", src=placed_sdf,
    constraint_smarts=CORE_SMARTS, constraint_weight=25.0, basin_hops=0,
)
rmsds = [core_rmsd_to_crystal(m) for m in con_mols]
print(f"constrained dock: {len(con_mols)} poses in {con_t:.1f}s  "
      f"(mean score {scores(con_mols).mean():.2f})")
print(f"core RMSD to crystal: max {max(rmsds):.3f} A  "
      f"(every core pinned to the known pose)")
n_starts_used = {m.GetIntProp("docking_n_starts_used") for m in con_mols}
print("n_starts actually used:", n_starts_used)

## (c) Scaffold-indexed docking: automate it across the series

Placing a template by hand for every molecule is exactly what scaffold indexing
automates. With `index_poses=True`:

- The dock block maintains a single-file cache at `./.cmxflow/scaffold_index.db`,
  keyed by Bemis–Murcko scaffold (and namespaced by the docking parameters).
- The **reference ligand's** scaffold pose is seeded first, so the whole series is
  a cache **hit** immediately: each congener transfers the cached core and runs a
  single constrained local search — no manual placement, no SMARTS.
- Cache writes are first-writer-wins (`INSERT OR IGNORE`), so it is safe across the
  parallel workers, and the cache persists across runs.

Each docked pose is tagged with a `docking_indexed` provenance flag.

In [ ]:
# Start from a cold cache so the demo is reproducible; the cache would normally
# persist and be reused (and auto-discovered) on later runs.
shutil.rmtree(".cmxflow", ignore_errors=True)

idx_path, idx_mols, idx_t = run_dock(
    "docked_indexed.sdf", index_poses=True, basin_hops=0
)

templated = sum(
    1 for m in idx_mols if m.GetProp("docking_indexed") in ("1", "True", "true")
)
print(f"indexed dock: {len(idx_mols)} poses in {idx_t:.1f}s  "
      f"(mean score {scores(idx_mols).mean():.2f})")
print(f"templated (cache hits): {templated}/{len(idx_mols)}")
print(f"cache file: {Path('.cmxflow/scaffold_index.db').resolve()} "
      f"({Path('.cmxflow/scaffold_index.db').stat().st_size} bytes)")
print(f"\nspeedup vs blind dock: {blind_t / idx_t:.1f}x  "
      f"({blind_t:.1f}s -> {idx_t:.1f}s for the same {len(idx_mols)} molecules)")

## Summary

In [ ]:
summary = pd.DataFrame([
    {"mode": "(a) blind", "wall_s": round(blind_t, 1),
     "mean_score": round(scores(blind_mols).mean(), 2), "output": blind_path.name},
    {"mode": "(a) blind + ILS", "wall_s": round(ils_t, 1),
     "mean_score": round(scores(ils_mols).mean(), 2), "output": ils_path.name},
    {"mode": "(b) constrained", "wall_s": round(con_t, 1),
     "mean_score": round(scores(con_mols).mean(), 2), "output": con_path.name},
    {"mode": "(c) indexed", "wall_s": round(idx_t, 1),
     "mean_score": round(scores(idx_mols).mean(), 2), "output": idx_path.name},
])
print("Docked-pose SDFs written to", OUT.resolve())
summary

In [ ]:
# Tidy up scratch inputs and the demo cache; keep only the docked-pose SDFs.
shutil.rmtree(SCRATCH, ignore_errors=True)
shutil.rmtree(".cmxflow", ignore_errors=True)
print("done")